In [52]:
from google.colab import userdata
import os, google.generativeai as genai

API_KEY = userdata.get('GOOGLE_API_KEY_1')
if not API_KEY:
    print("No secret found. 请到 Colab User secrets 里设置 GOOGLE_API_KEY")
else:
    genai.configure(api_key=API_KEY)


# 医美企业 AI 陪练对话：数据处理与指标构建  
林珑

本报告对企业内部的“员工–AI 陪练”对话数据进行结构化处理，将多层 JSON 展开为消息级与练习级两张表，并在此基础上构建练习频率/节奏、互动结构、文本风格与“对话新颖性”等指标。新颖性同时采用**词面法（TF-IDF/Jaccard）**与**语义法（Gemini Embedding）**两种口径，以提升对“表达路径变化”的刻画能力。全流程以程序实现，可复现并导出**单一汇总 CSV**，为成效评估与训练策略优化提供数据基础。

## 1. 研究问题与方法

**目标**  
从“员工-AI”陪练对话中抽取练习行为特征（频率/节奏、互动结构、文本风格、新颖性），为后续的效果评估与训练策略优化提供数据底座。

**方法**  
将 JSON 拉平成两张表（消息级、练习级），构建如下指标并做统计/可视化：

- **互动结构**  
  - `turns_total / turns_user / turns_assistant`  
  - `role_balance_user_over_asst`（用户轮次 ÷ AI 轮次）

- **文本特征**  
  - 词数/字数：`avg_len_tokens_*`、`avg_len_chars_*`（全体/用户/AI）  
  - 提问率：`question_rate_*`（全体/用户/AI）  
  - 词汇多样性（词表层面）：`ttr_all / ttr_user / ttr_asst`

- **频率与节奏**  
  - 时间字段：`practice_dt`  
  - 与上次练习间隔：`gap_since_prev_hours`  
  - 频率统计：`daily_practice_count`、`weekly_practice_count`  
  - 学习轨迹：`session_idx`（员工内第几次）  
  - 分数走势：`score_delta_prev`（与上次分数差）、`score_ma3`（分数 3 次滚动均值）

- **对话新颖性**  
  - 词面版：`novelty_1_minus_maxsim`（TF-IDF / Jaccard，越大越新）  
  - 语义版：`novelty_embed_gemini_1_minus_maxsim`（Gemini Embedding）  
  



## 2. 数据来源与结构

**数据对象**：一次练习（`practice_id`）包含若干消息（`chat_logs`）。  
**主要字段**：

practice_id：练习唯一 ID

practice_time：开始时间（精确到秒）

practice_duration：持续时长（毫秒）

employee_uid / employee_name / employee_department：员工标识

subscene_info：场景信息（title、description、objective）

practice_results：结果（total_score、evaluation 文本评语，有部分缺失）

practice_conversation.chat_logs：对话轮次列表  




In [32]:
import os, json, math
from datetime import datetime
from collections import defaultdict, Counter
import pandas as pd


try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

DATA_PATH = "practice_data.json"
OUTPUT_CSV = "processed_conversations.csv"

DATA_PATH, SKLEARN_AVAILABLE


('practice_data.json', True)

In [33]:

# 预览首条记录的关键字段
sample = records[0]
list(sample.keys()), sample.get("practice_id"), sample.get("employee_name"), sample.get("subscene_info", {})


(['practice_id',
  'practice_time',
  'practice_duration',
  'employee_uid',
  'employee_name',
  'employee_department',
  'subscene_info',
  'practice_results',
  'practice_conversation'],
 12522,
 '姚娟',
 {'title': '向第一次来的客户推荐办卡',
  'description': '建立信任，推荐会员卡和疗程卡，促成购买',
  'objective': '建立信任，使客户对百丽雅的会员卡和疗程卡产生兴趣，并考虑购买。'})

## 3. 展开原始数据与基础特征

**目的**：将多层 JSON（一次练习=多轮消息）结构化为两张表，并在练习级别抽取基础特征，为后续“节奏/频率、新颖性、成效评估”奠定数据基础。  
- **消息级** `message_df`：每条消息一行，包含 `practice_id / role / content / turn_index / time`；用于相似度、Embedding 等 NLP 计算。  
- **练习级** `df`：每个 `practice_id` 一行，抽取互动结构、文本长度、提问率、词汇多样性（TTR）、角色均衡等特征。

**练习级特征**  
- 互动结构：总轮次/用户轮次/AI轮次与角色均衡 `turns_total`, `turns_user`, `turns_assistant`, `role_balance_user_over_asst = turns_user / max(1, turns_assistant)`  
- 文本长度：平均词数与平均字数 `avg_len_tokens_*`, `avg_len_chars_*`, 分别统计 *all/user/asst*
- 提问率：含“?”或“？”句子比例 `question_rate_*`, 分别统计 *all/user/asst*  
- 词汇多样性：去重词数 / 总词数 `ttr_* = unique_tokens / tokens`, 分别统计 *all/user/asst*
- 结果字段：`total_score`, `evaluation`（原数据可能缺失，保持缺失不做臆填）


In [34]:
# 工具函数：判断问句 & 词汇多样性
def is_question(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return "？" in text or "?" in text

def uniq_ratio(tokens):
    if not tokens:
        return 0.0
    return len(set(tokens)) / max(1, len(tokens))


In [35]:
import pandas as pd

def build_tables(records):
    practice_rows = []
    message_rows = []

    for rec in records:
        pid = rec.get("practice_id")
        ptime = rec.get("practice_time")
        pduration_ms = rec.get("practice_duration")
        uid = rec.get("employee_uid")
        ename = rec.get("employee_name")
        dept = rec.get("employee_department")

        scene = rec.get("subscene_info", {}) or {}
        results = rec.get("practice_results", {}) or {}
        chat_logs = (rec.get("practice_conversation", {}) or {}).get("chat_logs", [])

        # —— 消息级：逐条保留
        for i, m in enumerate(chat_logs):
            message_rows.append({
                "practice_id": pid,
                "employee_uid": uid,
                "employee_name": ename,
                "time": ptime,
                "turn_index": i,
                "role": m.get("role"),
                "content": m.get("content", "")
            })

        # —— 练习级：聚合统计
        n_turns = len(chat_logs)
        n_user  = sum(1 for m in chat_logs if m.get("role") == "USER")
        n_asst  = sum(1 for m in chat_logs if m.get("role") == "ASSISTANT")

        all_text  = " ".join(m.get("content", "") for m in chat_logs)
        user_text = " ".join(m.get("content", "") for m in chat_logs if m.get("role") == "USER")
        asst_text = " ".join(m.get("content", "") for m in chat_logs if m.get("role") == "ASSISTANT")

        tok_all  = all_text.split()
        tok_user = user_text.split()
        tok_asst = asst_text.split()

        practice_rows.append({
            "practice_id": pid,
            "practice_time": ptime,
            "practice_duration_min": round((pduration_ms or 0)/60000.0, 3),

            "employee_uid": uid,
            "employee_name": ename,
            "employee_department": dept,

            "scene_title": scene.get("title"),
            "scene_description": scene.get("description"),
            "scene_objective": scene.get("objective"),

            # 互动结构
            "turns_total": n_turns,
            "turns_user": n_user,
            "turns_assistant": n_asst,
            "role_balance_user_over_asst": round(n_user/max(1,n_asst), 3),

            # 文本长度（词/字）
            "avg_len_tokens_all": round(len(tok_all)/n_turns if n_turns else 0.0, 3),
            "avg_len_tokens_user": round(len(tok_user)/max(1,n_user), 3),
            "avg_len_tokens_asst": round(len(tok_asst)/max(1,n_asst), 3),

            "avg_len_chars_all": round(len(all_text)/n_turns if n_turns else 0.0, 3),
            "avg_len_chars_user": round(len(user_text)/max(1,n_user), 3),
            "avg_len_chars_asst": round(len(asst_text)/max(1,n_asst), 3),

            # 提问率
            "question_rate_all": round(sum(1 for m in chat_logs if is_question(m.get("content","")))/max(1,n_turns), 3),
            "question_rate_user": round(sum(1 for m in chat_logs if m.get("role")=="USER" and is_question(m.get("content","")))/max(1,n_user), 3),
            "question_rate_asst": round(sum(1 for m in chat_logs if m.get("role")=="ASSISTANT" and is_question(m.get("content","")))/max(1,n_asst), 3),

            # 词汇多样性
            "ttr_all":  round(uniq_ratio(tok_all), 3),
            "ttr_user": round(uniq_ratio(tok_user), 3),
            "ttr_asst": round(uniq_ratio(tok_asst), 3),

            # 结果字段（可能缺失）
            "total_score": (results or {}).get("total_score"),
            "evaluation": (results or {}).get("evaluation"),
        })

    df_practice = pd.DataFrame(practice_rows)
    df_message  = pd.DataFrame(message_rows)
    return df_practice, df_message

# —— 调用：把 records 变成两张表
df, message_df = build_tables(records)
df.head(3)


,practice_id,practice_time,practice_duration_min,employee_uid,employee_name,employee_department,scene_title,scene_description,scene_objective,turns_total,...,avg_len_chars_user,avg_len_chars_asst,question_rate_all,question_rate_user,question_rate_asst,ttr_all,ttr_user,ttr_asst,total_score,evaluation
0,12522,2023-11-10 10:01:50,3.174,1326,姚娟,百丽雅/购物公园店,向第一次来的客户推荐办卡,建立信任，推荐会员卡和疗程卡，促成购买,建立信任，使客户对百丽雅的会员卡和疗程卡产生兴趣，并考虑购买。,21,...,81.200,41.545,0.333,0.300,0.364,1.0,1.0,1.0,87.15,非常棒，冗余词占比低于4%，能让语言表达更加直接和简洁。
1,12534,2023-11-10 10:25:28,5.226,1347,清凤,百丽雅,向第一次来的客户推荐办卡,建立信任，推荐会员卡和疗程卡，促成购买,建立信任，使客户对百丽雅的会员卡和疗程卡产生兴趣，并考虑购买。,27,...,122.077,35.857,0.333,0.231,0.429,1.0,1.0,1.0,84.32,非常棒，发音清晰度高，听众能够轻松理解你所表达的意思。
2,12542,2023-11-10 10:46:44,3.458,1396,吴倩,百丽雅/双美中心生美部,向第一次来的客户推荐办卡,建立信任，推荐会员卡和疗程卡，促成购买,建立信任，使客户对百丽雅的会员卡和疗程卡产生兴趣，并考虑购买。,15,...,126.714,28.750,0.333,0.000,0.625,1.0,1.0,1.0,84.31,你的平均语速在232.87字/分钟，非常棒👍，这样能让听众更好地理解和接收你所传达的信息。


## 4. 练习节奏与频率指标（时间解析、间隔、日/周频次）

**目标**  
刻画员工在时间维度上的**练习节奏**与**强度**，为“学习曲线”和“冲刺/低谷”识别提供依据。

**时间解析**  
将 `practice_time` 解析为 `practice_dt`（`datetime`），按 `employee_uid, practice_dt` 排序，作为时序计算基础。

**与上次练习的时间间隔**  
对同一员工 \(i\) 的按时间排序后的第 \(k\) 次练习，记时间为 \(t_i^{(k)}\)。定义：
$$
\text{gap_since_prev_hours_}i^{(k)}=
\begin{cases}
\text{NaN}, & k=1 \\
\frac{t_i^{(k)}-t_i^{(k-1)}}{\text{1 hour}}, & k>1
\end{cases}
$$
并给出首练标记：

\begin{cases}
\text{True}, & k=1\\
\text{False}, & k>1
\end{cases}


**日/周练习频次**  
- **日频**：将 `practice_dt` 取日期 \(d\)，统计员工 \(i\) 当日练习次数
- **周频**：取 ISO 年-周 \(y\!-\!w\)（此处使用 `strftime('%G-%V')`）
- 解析失败的时间置为 `NaT`；相关时序指标将返回缺失。  
- 首练的 `gap_since_prev_hours=None / NaN`，并置 `is_first_for_employee=True`。  
- `daily_practice_count` 通过 `groupby(employee_uid, date).size()` 计算后回并；  
  `weekly_practice_count` 通过 `groupby(employee_uid, year_week).size()` 计算后回并。  


In [36]:
from datetime import datetime

def parse_time(s):
    try:
        return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
    except Exception:
        return pd.NaT

df["practice_dt"] = df["practice_time"].apply(parse_time)
df.sort_values(["employee_uid","practice_dt"], inplace=True)

# 与上次练习的时间间隔（小时）
df["gap_since_prev_hours"] = None
df["is_first_for_employee"] = False

for uid, sub in df.groupby("employee_uid"):
    idxs = sub.index.tolist()
    prev = None
    for i, idx in enumerate(idxs):
        cur = df.loc[idx, "practice_dt"]
        if i == 0 or pd.isna(cur) or prev is None:
            df.loc[idx, "gap_since_prev_hours"] = None
            df.loc[idx, "is_first_for_employee"] = True
        else:
            delta = cur - prev
            df.loc[idx, "gap_since_prev_hours"] = round(delta.total_seconds()/3600.0, 3)
        prev = cur

# 日/周练习频率
df["date"] = df["practice_dt"].dt.date
daily = df.groupby(["employee_uid","date"]).size().reset_index(name="daily_practice_count")
df = df.merge(daily, on=["employee_uid","date"], how="left")

df["year_week"] = df["practice_dt"].dt.strftime("%G-%V")
weekly = df.groupby(["employee_uid","year_week"]).size().reset_index(name="weekly_practice_count")
df = df.merge(weekly, on=["employee_uid","year_week"], how="left")

df[["employee_name","practice_time","daily_practice_count","weekly_practice_count","gap_since_prev_hours","is_first_for_employee"]].head(5)


,employee_name,practice_time,daily_practice_count,weekly_practice_count,gap_since_prev_hours,is_first_for_employee
0,陈香,2023-11-10 20:05:55,2,2,None,True
1,陈香,2023-11-10 20:13:44,2,2,0.13,False
2,姚娟,2023-11-10 10:01:50,2,2,None,True
3,姚娟,2023-11-10 14:46:40,2,2,4.747,False
4,王易梦,2023-11-10 20:09:59,1,1,None,True



## 5. 分数提升与滚动均值

**目的**  
展示“越练越好”的学习轨迹，并给出去噪后的趋势线。

**定义**  
- `session_idx`：员工内第几次练习（按 `practice_dt` 排序后从 1 计数）。  
- `score_delta_prev`：与上一场的分数差，\(\Delta s_t = s_t - s_{t-1}\)。  
- `score_ma3`：分数 3 次滚动均值（员工内；不足 3 次时取已有均值）。




In [44]:
import pandas as pd
import numpy as np

# 转数值分数 & 时间确保可排序
df["total_score_num"] = pd.to_numeric(df.get("total_score"), errors="coerce")
df["practice_dt"] = pd.to_datetime(df.get("practice_time"), errors="coerce")

# 员工内时间排序 + 序号
df = df.sort_values(["employee_uid","practice_dt"])
df["session_idx"] = df.groupby("employee_uid").cumcount() + 1

# 分数提升（上一场 vs 当前场）
df["score_delta_prev"] = df.groupby("employee_uid")["total_score_num"].diff()


# 3 次滚动均值（员工内；不足 3 次也给均值）
_rolling3 = lambda s: s.rolling(3, min_periods=1).mean()
df["score_ma3"] = df.groupby("employee_uid")["total_score_num"].transform(_rolling3)



## 6. 词面新颖性（TF-IDF / Jaccard）

**目的**  
衡量同一员工当前练习与其**历史**练习的“词面差异”。数值∈[0,1]，**越大越新**。

**文本构造**  
将一次练习的消息按顺序拼接为全文 `by_pid_text[p]`（可选“仅 USER 侧”或“全体消息”）。

**仅在员工内比较**  
历史集合：
$$
\mathcal{H}_i(t)=\{\,p' \mid emp(p')=i,\; time(p')<t\,\}
$$

**相似度与定义**  
优先使用 **TF-IDF 余弦相似度**，不可用时回退 **Jaccard**（词集合交并比）。  
新颖性（取“与历史最像的一次”并取反）：
$$
N_{\text{lex}}(p)=1-\max_{p'\in \mathcal{H}_i(t)} s(p,p')
$$

- TF-IDF 示例超参：`min_df=2, max_df=0.95, token_pattern=r"(?u)\b\w+\b"`.
- Jaccard：\( J(A,B)=\frac{|A\cap B|}{|A\cup B|} \).
- 高值（≈1）：表达/措辞明显变化；低值（≈0）：高度复用旧表达。


In [37]:
# 构造每次练习的全文本
by_pid_text = {}
for r in message_rows:
    by_pid_text.setdefault(r["practice_id"], []).append(r.get("content",""))
for k in list(by_pid_text.keys()):
    by_pid_text[k] = "\n".join(by_pid_text[k]).strip()

novelty = []
for idx, row in df.iterrows():
    uid = row["employee_uid"]
    pid = row["practice_id"]
    cur_text = by_pid_text.get(pid, "")
    t = row["practice_dt"]
    prior_pids = df[(df["employee_uid"]==uid) & (df["practice_dt"] < t)]["practice_id"].tolist()

    if not prior_pids:
        novelty.append(1.0)
        continue

    prior_texts = [by_pid_text.get(pp,"") for pp in prior_pids]
    if SKLEARN_AVAILABLE:
        vect = TfidfVectorizer(min_df=1, max_df=0.9)
        X = vect.fit_transform(prior_texts + [cur_text])
        from sklearn.metrics.pairwise import cosine_similarity
        sims = cosine_similarity(X[-1], X[:-1]).flatten()
        max_sim = float(sims.max()) if sims.size else 0.0
    else:
        cur_set = set(cur_text.split())
        sims = []
        for pt in prior_texts:
            s = set(pt.split())
            inter = len(cur_set & s)
            uni = len(cur_set | s) if (cur_set or s) else 1
            sims.append(inter/uni if uni else 0.0)
        max_sim = max(sims) if sims else 0.0

    novelty.append(round(1.0 - max_sim, 4))

df["novelty_1_minus_maxsim"] = novelty
df[["employee_name","practice_time","novelty_1_minus_maxsim"]].head(5)


,employee_name,practice_time,novelty_1_minus_maxsim
0,陈香,2023-11-10 20:05:55,1.0
1,陈香,2023-11-10 20:13:44,1.0
2,姚娟,2023-11-10 10:01:50,1.0
3,姚娟,2023-11-10 14:46:40,1.0
4,王易梦,2023-11-10 20:09:59,1.0


## 7. 语义新颖性（Gemini Embedding）

**目的**  
解决词面法“换个说法就显新”的问题，刻画**表达路径/意图**是否变化。数值∈[0,1]，**越大越新**。

**向量化**  
用 **Gemini `text-embedding-004`** 将 `by_pid_text[p]` 编码为向量 \( \mathbf{e}_p \)。

**仅在员工内比较**  
历史集合同上：
$$
\mathcal{H}_i(t)=\{\,p' \mid emp(p')=i,\; time(p')<t\,\}
$$

**相似度与定义**  
采用余弦相似度：
$$
s_{\cos}(p,p')=\frac{\mathbf{e}_p\cdot\mathbf{e}_{p'}}{\|\mathbf{e}_p\|\,\|\mathbf{e}_{p'}\|}
$$
语义新颖性（取“与历史最像的一次”并取反）：
$$
N_{\text{sem}}(p)=1-\max_{p'\in \mathcal{H}_i(t)} s_{\cos}(p,p')
$$


In [38]:
# Gemini Embedding 新颖性（语义向量）
import time, numpy as np, google.generativeai as genai

EMBED_MODEL = "text-embedding-004"

def get_gemini_embedding(text: str):
    if not text or not text.strip():
        return None
    r = genai.embed_content(model=EMBED_MODEL, content=text)
    return r["embedding"]

# by_pid_text
pid2vec = {}
for pid, txt in by_pid_text.items():
    try:
        pid2vec[pid] = get_gemini_embedding(txt)
        time.sleep(0.05)
    except Exception as e:
        print("embed error:", pid, e)
        pid2vec[pid] = None

def cos(a, b):
    a, b = np.array(a), np.array(b)
    if a.size==0 or b.size==0: return 0.0
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na==0 or nb==0: return 0.0
    return float(np.dot(a, b) / (na * nb))

novelty_embed = []
for _, row in df.iterrows():
    uid, pid, t = row["employee_uid"], row["practice_id"], row["practice_dt"]
    vec = pid2vec.get(pid)
    prior = df[(df["employee_uid"]==uid) & (df["practice_dt"]<t)]["practice_id"].tolist()
    if not prior or vec is None:
        novelty_embed.append(1.0); continue
    sims = [cos(vec, pid2vec.get(pp)) for pp in prior if pid2vec.get(pp) is not None]
    max_sim = max(sims) if sims else 0.0
    novelty_embed.append(round(1.0 - max_sim, 4))

df["novelty_embed_gemini_1_minus_maxsim"] = novelty_embed
df[["practice_id", "novelty_1_minus_maxsim", "novelty_embed_gemini_1_minus_maxsim"]].head()


,practice_id,novelty_1_minus_maxsim,novelty_embed_gemini_1_minus_maxsim
0,12834,1.0,1.0000
1,12837,1.0,0.1683
2,12522,1.0,1.0000
3,12688,1.0,0.1006
4,12835,1.0,1.0000



## 8. 互动与可读性

**交替率 `alternation_rate`**  
员工与 AI 角色**切换次数 / 轮次**，衡量对话是否“你一句我一句”的互动节奏。  
- 取值范围：\[0, 1]；越高互动越频繁；极低可能是长段独白。

**AI 词占比 `assistant_token_share`**  
AI 侧词数 ÷（用户词数 + AI 词数），衡量**AI 话语主导度**。  
- 取值范围：\[0, 1]；>0.6 可能表示 AI 输出过重，<0.4 可能表示用户“倒灌信息”。

**AI 相邻重复 `asst_adjacent_jaccard_mean`**  
AI 相邻两轮**词集合 Jaccard 相似度**的均值，衡量**模板化/重复感**。  
- 取值范围：\[0, 1]；越高越“像在重复”；≈NaN 表示只有 0/1 条 AI 发言，无法计算。


In [55]:
from collections import defaultdict
import numpy as np

# 先按 practice_id 取回整场对话（确保按 turn_index 排序）
by_pid_chat = defaultdict(list)
for r in message_df.sort_values(["practice_id","turn_index"]).to_dict("records"):
    by_pid_chat[r["practice_id"]].append(r)

def alternation_rate(chat):
    """角色切换次数 / 轮次"""
    sw, last = 0, None
    for m in chat:
        role = m.get("role")
        if last is not None and role != last:
            sw += 1
        last = role
    return round(sw / max(1, len(chat)), 3)

def assistant_token_share(chat):
    """AI 词占比 = AI词数 / (USER词数 + AI词数)"""
    u = sum(len((m.get("content") or "").split()) for m in chat if m.get("role")=="USER")
    a = sum(len((m.get("content") or "").split()) for m in chat if m.get("role")=="ASSISTANT")
    return round(a / max(1, u + a), 3)

def jaccard(a: str, b: str) -> float:
    """词集合 Jaccard 相似度"""
    A, B = set((a or "").split()), set((b or "").split())
    return 0.0 if not A or not B else len(A & B) / max(1, len(A | B))

# 逐练习计算三项指标
alts, shares, rep = [], [], []
for pid in df["practice_id"]:
    chat = by_pid_chat.get(pid, [])

    # 交替率 / AI词占比
    alts.append(alternation_rate(chat))
    shares.append(assistant_token_share(chat))

    # AI 相邻重复（只有 >=2 条 AI 发言才计算）
    ai_utts = [(m.get("content") or "") for m in chat if m.get("role")=="ASSISTANT"]
    sims = [jaccard(ai_utts[i-1], ai_utts[i]) for i in range(1, len(ai_utts))] if len(ai_utts) >= 2 else []
    rep.append(round(float(np.mean(sims)), 3) if sims else np.nan)

df["alternation_rate"] = alts
df["assistant_token_share"] = shares
df["asst_adjacent_jaccard_mean"] = rep



In [56]:
import pandas as pd

df_export = df.copy()

# 时间列转成字符串
if "practice_dt" in df_export.columns:
    df_export["practice_dt"] = df_export["practice_dt"].dt.strftime("%Y-%m-%d %H:%M:%S")

out_csv = "processed_conversations.csv"
df_export.to_csv(out_csv, index=False, encoding="utf-8-sig")  # utf-8-sig 适配 Excel

print(f"已导出：{out_csv}  | 行×列 = {df_export.shape}")


已导出：processed_conversations.csv  | 行×列 = (45, 45)
